In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

%config InlineBackend.figure_format = 'svg'

In [ ]:
# ============================================================
#
# This script is intended for system-level jitter budgeting.
#
# ============================================================
# MODELING ASSUMPTIONS
# ============================================================
#
# 1) All jitter quantities are RMS values.
# 2) Additive jitter contributions are assumed statistically
#    independent and are combined using root-sum-square (RSS).
# 3) Only random (RJ) components are modeled.
#    Deterministic jitter (DJ) is not included.
# 4) Supply-induced jitter, radiation effects, temperature drift,
#    aging, EMI, PCB coupling, and correlation effects are excluded.
# 5) This model evaluates clock-limited SNR only.
#    ADC quantization noise, thermal noise, and analog front-end
#    noise are not included.
#
# ============================================================

# ============================================
# Load Configuration
# ============================================
with open("config.yaml", "r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

# ============================================
# Output Directory Setup
# ============================================

REPORT_DIR = Path(config["paths"]["report_directory"])
REPORT_DIR.mkdir(exist_ok=True)

print(f"Reports directory ready at: {os.path.abspath(REPORT_DIR)}")

In [ ]:
# Extract jitter parameters
sigma_osc_typ = float(config["sigma_osc_typ"])
sigma_osc_max = float(config["sigma_osc_max"])
sigma_fanout_typ = float(config["sigma_fanout_typ"])
sigma_fanout_max = float(config["sigma_fanout_max"])
sigma_buffer = float(config["sigma_buffer"])
sigma_adc = float(config["sigma_adc"])
f0 = float(config["f0"])

# ------------------------------------------------------------
# Requirement Targets (TIM-001 / SYS-001)
# ------------------------------------------------------------
ENOB_required = 10.5
SNR_required = 6.02 * ENOB_required + 1.76
# ------------------------------------------------------------

# ------------------------------------------------------------
# RSS COMBINATION OF JITTER SOURCES
# ------------------------------------------------------------

sigma_total_typ = np.sqrt(
    sigma_osc_typ**2 + sigma_fanout_typ**2 + sigma_buffer**2 + sigma_adc**2
)

sigma_total_max = np.sqrt(
    sigma_osc_max**2 + sigma_fanout_max**2 + sigma_buffer**2 + sigma_adc**2
)

print("\n--- Jitter Contributions (Typical Oscillator) ---")
print(f"Oscillator: {sigma_osc_typ * 1e12:.3f} ps")
print(f"Fanout: {sigma_fanout_typ * 1e12:.3f} ps")
print(f"LVDS->CMOS: {sigma_buffer * 1e12:.3f} ps")
print(f"ADC aperture: {sigma_adc * 1e12:.3f} ps")
print(f"Total RMS: {sigma_total_typ * 1e12:.3f} ps")

print("\n--- Jitter Contributions (Maximum Oscillator) ---")
print(f"Oscillator: {sigma_osc_max * 1e12:.3f} ps")
print(f"Fanout: {sigma_fanout_max * 1e12:.3f} ps")
print(f"LVDS->CMOS: {sigma_buffer * 1e12:.3f} ps")
print(f"ADC aperture: {sigma_adc * 1e12:.3f} ps")
print(f"Total RMS: {sigma_total_max * 1e12:.3f} ps")

# ------------------------------------------------------------
# PERFORMANCE AT CRITICAL OPERATING POINTS
# ------------------------------------------------------------

fs_1 = f0  # Hz
fs_2 = 65e6  # Hz

f_nyq_1 = fs_1 / 2  # Hz
f_nyq_2 = fs_2 / 2  # Hz

# --- Typical ---
SNR_40_typ = -20 * np.log10(2 * np.pi * f_nyq_1 * sigma_total_typ)
ENOB_40_typ = (SNR_40_typ - 1.76) / 6.02

SNR_65_typ = -20 * np.log10(2 * np.pi * f_nyq_2 * sigma_total_typ)
ENOB_65_typ = (SNR_65_typ - 1.76) / 6.02

# --- Maximum ---
SNR_40_max = -20 * np.log10(2 * np.pi * f_nyq_1 * sigma_total_max)
ENOB_40_max = (SNR_40_max - 1.76) / 6.02

SNR_65_max = -20 * np.log10(2 * np.pi * f_nyq_2 * sigma_total_max)
ENOB_65_max = (SNR_65_max - 1.76) / 6.02

print("\n--- Jitter-Limited Performance at Nyquist (Typical) ---")
print(f"40 MSPS: SNR = {SNR_40_typ:.2f} dB | ENOB = {ENOB_40_typ:.2f} bits")
print(f"65 MSPS: SNR = {SNR_65_typ:.2f} dB | ENOB = {ENOB_65_typ:.2f} bits")

print("\n--- Jitter-Limited Performance at Nyquist (Maximum) ---")
print(f"40 MSPS: SNR = {SNR_40_max:.2f} dB | ENOB = {ENOB_40_max:.2f} bits")
print(f"65 MSPS: SNR = {SNR_65_max:.2f} dB | ENOB = {ENOB_65_max:.2f} bits")

# ------------------------------------------------------------
# JITTER-LIMITED SNR AND ENOB MODEL
# ------------------------------------------------------------

f_in = np.logspace(3, np.log10(f_nyq_2), 2000)

SNR_jitter_typ = -20 * np.log10(2 * np.pi * f_in * sigma_total_typ)
ENOB_jitter_typ = (SNR_jitter_typ - 1.76) / 6.02

SNR_jitter_max = -20 * np.log10(2 * np.pi * f_in * sigma_total_max)
ENOB_jitter_max = (SNR_jitter_max - 1.76) / 6.02

# ------------------------------------------------------------
# High-Resolution SNR Plot (SVG Export)
# ------------------------------------------------------------

plt.figure(figsize=(12, 8))

plt.semilogx(f_in, SNR_jitter_typ, linewidth=2)
plt.semilogx(f_in, SNR_jitter_max, "--", linewidth=2)

# Required SNR horizontal line
plt.plot([np.min(f_in), np.max(f_in)], [SNR_required, SNR_required], "r:", linewidth=2)

# Get current y-axis limits before adding vertical lines
y_lim = plt.ylim()

# Nyquist vertical lines
plt.plot([f_nyq_1, f_nyq_1], y_lim, "k--", linewidth=1.5)
plt.plot([f_nyq_2, f_nyq_2], y_lim, "k-.", linewidth=1.5)

plt.grid(True)
plt.tick_params(labelsize=14)

plt.xlabel("Input Frequency (Hz)", fontsize=16)
plt.ylabel("Jitter-Limited SNR (dB)", fontsize=16)
plt.title("Clock-Limited SNR vs Input Frequency", fontsize=18)

plt.legend(
    ["Typical", "Maximum", "Required SNR", "Nyquist 40 MSPS", "Nyquist 65 MSPS"],
    loc="lower left",
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        REPORT_DIR,
        "ANA005_xx_SNRJitter.svg",
    )
)
plt.show()

# ------------------------------------------------------------
# High-Resolution ENOB Plot (SVG Export)
# ------------------------------------------------------------

plt.figure(figsize=(12, 8))

plt.semilogx(f_in, ENOB_jitter_typ, linewidth=2)
plt.semilogx(f_in, ENOB_jitter_max, "--", linewidth=2)

# Required ENOB horizontal line
plt.plot(
    [np.min(f_in), np.max(f_in)], [ENOB_required, ENOB_required], "r:", linewidth=2
)

# Get current y-axis limits before adding vertical lines
y_lim = plt.ylim()

# Nyquist vertical lines
plt.plot([f_nyq_1, f_nyq_1], y_lim, "k--", linewidth=1.5)
plt.plot([f_nyq_2, f_nyq_2], y_lim, "k-.", linewidth=1.5)

plt.grid(True)
plt.tick_params(labelsize=14)

plt.xlabel("Input Frequency (Hz)", fontsize=16)
plt.ylabel("Effective Number of Bits (ENOB)", fontsize=16)
plt.title("Clock-Limited ENOB vs Input Frequency", fontsize=18)

plt.legend(
    ["Typical", "Maximum", "Required ENOB", "Nyquist 40 MSPS", "Nyquist 65 MSPS"],
    loc="lower left",
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        REPORT_DIR,
        "ANA005_xx_ENOBJitter.svg",
    )
)
plt.show()

# ------------------------------------------------------------
# JITTER CONTRIBUTION PIE CHART (Typical Case)
# ------------------------------------------------------------
#
# Jitter adds in variance (sigma^2).
# Contribution percentage is computed from
# variance ratio, not sigma ratio.
#

# Variance contributions (typical case)
var_osc = sigma_osc_typ**2
var_fanout = sigma_fanout_typ**2
var_buffer = sigma_buffer**2
var_adc = sigma_adc**2

var_total = sigma_total_typ**2

# Percentage contribution
pct = 100 * np.array([var_osc, var_fanout, var_buffer, var_adc]) / var_total

plt.figure(figsize=(12, 8))
plt.pie(pct)
plt.title("Jitter Variance Contribution (Typical Case)", fontsize=16)
plt.legend(
    [
        f"Oscillator ({pct[0]:.1f}%)",
        f"Fanout ({pct[1]:.1f}%)",
        f"LVDS->CMOS ({pct[2]:.1f}%)",
        f"ADC Aperture ({pct[3]:.1f}%)",
    ],
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=14,
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        REPORT_DIR,
        "ANA005_xx_JitterPieTypical.svg",
    )
)


plt.show()